In [ ]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/raw/srodmiescie_raw_object_id.csv", sep=";", index_col=0, dtype=column_types
).dropna(subset="ground_floor_function")

data_gdynia.head()

1.

In [ ]:
group = data_gdynia.groupby("addr_street")
group.ngroups
group.get_group("Świętego Piotra").head(3)

In [ ]:
assert group.ngroups == data_gdynia["addr_street"].nunique()

2.

In [ ]:
count = data_gdynia.groupby("addr_street")["addr_number"].agg("count")
size = data_gdynia.groupby("addr_street")["addr_number"].agg("size")
diff = count - size
diff[diff != 0].head(5)

In [ ]:
a = data_gdynia["addr_street"].value_counts().rename("addr_number").sort_index()
b = (
    data_gdynia.groupby("addr_street")["addr_number"]
    .agg("size")
    .sort_values(ascending=False)
    .sort_index()
)

assert a.equals(b)

3.

In [ ]:
data_gdynia.groupby(["addr_street", "condition"])["osm_id"].count().unstack().head()

4.

In [ ]:
data_gdynia.groupby("addr_street")["condition"].agg(
    ["min", "max", "mean", "median"]
).head()

5.

In [ ]:
data_gdynia = data_gdynia.assign(
    nfunc=lambda d: d["ground_floor_function"].str.count(",") + 1
)
data_gdynia.head()

In [ ]:
data_gdynia.groupby("nfunc", as_index=False).agg(
    n_buildings=("addr_street", "size"),
    diff_streets=("addr_street", "nunique"),
    n_fill_adress=("addr_number", "count"),
)


6.

In [ ]:
def min_max(d):
    return d.max() - d.min()


f1 = data_gdynia.groupby("addr_street")["nfunc"].agg(min_max)
f2 = data_gdynia.groupby("addr_street")["nfunc"].apply(lambda d: d.max() - d.min())

In [ ]:
assert f1.equals(f2)

7.

In [ ]:
n_on_street = data_gdynia.groupby("addr_street")["nfunc"].transform("size")
fin = n_on_street[n_on_street >= 10]
fin

In [ ]:
assert fin.min() >= 10

8.

In [ ]:
data_clean = data_gdynia.assign(
    ground_floor_function=lambda d: (
        d["ground_floor_function"].str.replace("_", " ").str.lower().str.split(",")
    )
)
data_long = data_gdynia.explode("ground_floor_function").assign(
    ground_floor_function=lambda d: d["ground_floor_function"].str.strip()
)

In [ ]:
data_long.groupby(["addr_street", "ground_floor_function"])[("osm_id")].agg(
    "count"
).reset_index().rename(columns={"osm_id": "count"})

In [ ]:
top_func = (
    data_long.groupby(["addr_street", "ground_floor_function"])[("osm_id")]
    .agg("count")
    .reset_index()
    .sort_values(["addr_street", "osm_id"], ascending=[True, False])
    .rename(columns={"osm_id": "n"})
)
top_func_3 = top_func.groupby("addr_street").head(3)

In [ ]:
mask = top_func_3.groupby("addr_street").agg("size") > 3
assert mask.sum() == 0

In [ ]:
def sort_3(df, n=3):
    return df.sort_values("n", ascending=False, kind="stable")[:n]


counts = (
    data_long.groupby(["addr_street", "ground_floor_function"])["osm_id"]
    .agg("count")
    .reset_index()
    .rename(columns={"osm_id": "n"})
)
top_func_3b = counts.groupby("addr_street").apply(sort_3).reset_index(level=0)

In [ ]:
assert top_func_3.equals(top_func_3b)

In [ ]:
data_clean = data_clean.assign(
    n_on_street=lambda d: d.groupby("addr_street")["osm_id"].transform("size")
)

9

In [ ]:
num_b_str = pd.cut(
    data_clean["n_on_street"],
    [0, 10, 50, 100, 200],
    labels=["small_str", "medium_str", "large_str", "main_str"],
).reset_index()
num_b_str.groupby("n_on_street").agg("size")

In [ ]:
mask = data_clean["addr_street"].isna()
assert (
    len(data_clean) == mask.sum() + num_b_str.groupby("n_on_street").agg("size").sum()
)

In [ ]:
data_clean = data_clean.assign(
    street_size=lambda d: pd.cut(
        d["n_on_street"],
        [0, 10, 50, 100, 200],
        labels=["small_str", "medium_str", "large_str", "main_str"],
    )
)
data_clean = data_clean.assign(is_multi=lambda d: d["nfunc"] > 1)

In [ ]:
print(f"no of multifunc buildings: {data_clean['is_multi'].sum()}")
print(
    f"share of multifunc buildings: {data_clean['is_multi'].sum() / len(data_clean):.2f}"
)

In [ ]:
data_clean.groupby("street_size")["is_multi"].agg(["count", "mean"])

In [ ]:
data_clean = data_clean.assign(mean_func_per_street = lambda d: d.groupby('addr_street')['nfunc'].transform('mean'))
data_clean = data_clean.assign(nfuncminusmean = lambda d: d['nfunc']-d['mean_func_per_street'])

In [ ]:
sums = data_clean.groupby('addr_street')['nfuncminusmean'].agg('sum')
assert np.isclose(sums, 0).all()

In [ ]:
data_clean